# Generated Summary vs Gold Length Analysis

This notebook compares Chinese generated-summary token lengths against the gold reference summaries.

Inputs:
- `outputs/direct/all_models/direct_all_models_100samples_seed42_results.json`: 300 direct-generation rows from QWEN, Gemma, and Aya-Expanse.
- `outputs/mbart_baseline/mbart_large50_100samples_seed42_results.json`: 100 mBART baseline rows.

The main outputs are model-level length summaries, per-row comparisons for the 300 direct rows, all-model comparisons across 400 generations, and a sample-level wide table.

In [ ]:
from pathlib import Path
import json
import os
import re
import unicodedata

import pandas as pd

In [ ]:
def find_project_root(start: Path = Path.cwd()) -> Path:
    """Find the repository root from a notebook or script working directory."""
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "outputs").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing outputs/.")


PROJECT_ROOT = find_project_root()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "length_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Keep plotting and font caches inside the project tree for portable notebook execution.
MPLCONFIGDIR = OUTPUT_DIR / "matplotlib_cache"
XDG_CACHE_HOME = OUTPUT_DIR / "xdg_cache"
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
XDG_CACHE_HOME.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))
os.environ.setdefault("XDG_CACHE_HOME", str(XDG_CACHE_HOME))

print("Project root:", PROJECT_ROOT)
print("Output dir:", OUTPUT_DIR)

## Source Files

In [ ]:
DIRECT_PATH = PROJECT_ROOT / "outputs/direct/all_models/direct_all_models_100samples_seed42_results.json"
MBART_PATH = PROJECT_ROOT / "outputs/mbart_baseline/mbart_large50_100samples_seed42_results.json"

MODEL_NAME_MAP = {
    "jjnhuang/mbart-large-50-en-dialogue-to-zh-summary": "mBART",
    "qwen3.5:9b": "QWEN",
    "gemma4:e4b": "Gemma",
    "aya-expanse:8b": "Aya-Expanse",
}

MODEL_ORDER = ["Gold / Reference", "mBART", "QWEN", "Gemma", "Aya-Expanse"]

In [ ]:
def load_json_records(path: Path) -> list[dict]:
    """Load records from a JSON list or a JSON dictionary containing a list."""
    if not path.exists():
        raise FileNotFoundError(path)

    with path.open(encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        for value in data.values():
            if isinstance(value, list) and all(isinstance(item, dict) for item in value):
                return value
        return [data]

    raise TypeError(f"Unsupported JSON structure in {path}")


direct_records = load_json_records(DIRECT_PATH)
mbart_records = load_json_records(MBART_PATH)

print("Direct rows:", len(direct_records))
print("mBART rows:", len(mbart_records))

## Tokenization

`jieba` is used when available. If it is not installed, the fallback tokenizer counts each CJK character, each contiguous Latin/number span, and each placeholder-like token while excluding punctuation and symbols.

In [ ]:
try:
    import jieba

    TOKENIZER_NAME = "jieba.lcut"
except ModuleNotFoundError:
    jieba = None
    TOKENIZER_NAME = "regex fallback: CJK char / Latin span / number span / placeholder"


TOKEN_RE = re.compile(
    r"<[^>\s]+>|[A-Za-z]+(?:[-'][A-Za-z]+)*|\d+(?:\.\d+)?|[\u3400-\u9fff]|[^\s]"
)


def is_countable_token(token: str) -> bool:
    """Exclude whitespace, punctuation-only tokens, and symbol-only tokens."""
    token = str(token).strip()
    if not token:
        return False
    return any(not unicodedata.category(ch).startswith(("P", "S", "Z")) for ch in token)


def tokenize_summary(text: str) -> list[str]:
    """Tokenize a Chinese summary for length analysis."""
    text = str(text or "").strip()
    if not text:
        return []

    if jieba is not None:
        raw_tokens = jieba.lcut(text)
    else:
        raw_tokens = TOKEN_RE.findall(text)

    return [tok.strip() for tok in raw_tokens if is_countable_token(tok)]


def token_length(text: str) -> int:
    """Return the number of countable tokens in a summary."""
    return len(tokenize_summary(text))


print("Tokenizer:", TOKENIZER_NAME)
print("Example tokens:", tokenize_summary("佩顿给麦克斯提供了卖衣服的网站。"))

## Build the Comparison DataFrame

In [ ]:
def normalize_record(record: dict, source_file: str, source_group: str) -> dict:
    """Convert one raw generation record into a normalized length-comparison row."""
    raw_model_name = record.get("model_name", "")
    model = MODEL_NAME_MAP.get(raw_model_name, raw_model_name)

    generated = str(record.get("generated_summary_zh", "") or "").strip()
    gold = str(record.get("reference_summary_zh", "") or "").strip()

    generated_tokens = token_length(generated)
    gold_tokens = token_length(gold)

    return {
        "model": model,
        "raw_model_name": raw_model_name,
        "sample_index": record.get("sample_index"),
        "generated_summary_zh": generated,
        "reference_summary_zh": gold,
        "generated_tokens": generated_tokens,
        "gold_tokens": gold_tokens,
        "token_diff": generated_tokens - gold_tokens,
        "token_ratio": generated_tokens / gold_tokens if gold_tokens else None,
        "source_file": source_file,
        "source_group": source_group,
    }


rows = []

for record in direct_records:
    rows.append(normalize_record(record, DIRECT_PATH.name, "direct"))

for record in mbart_records:
    rows.append(normalize_record(record, MBART_PATH.name, "mbart_baseline"))

length_df = pd.DataFrame(rows)

display(length_df.head())
display(length_df["model"].value_counts().rename_axis("model").reset_index(name="n"))

In [ ]:
# Validate that the direct file contains 300 rows and the mBART file contains 100 rows.
assert len(direct_records) == 300, f"Expected 300 direct rows, found {len(direct_records)}"
assert len(mbart_records) == 100, f"Expected 100 mBART rows, found {len(mbart_records)}"

# Validate that each direct model has 100 rows.
direct_counts = length_df[length_df["source_group"] == "direct"]["model"].value_counts()
expected_direct_models = {"QWEN", "Gemma", "Aya-Expanse"}
assert set(direct_counts.index) == expected_direct_models, direct_counts.to_dict()
assert all(direct_counts == 100), direct_counts.to_dict()

# Validate that the same 100 sample indexes are shared by all model outputs.
samples_by_model = length_df.groupby("model")["sample_index"].apply(lambda s: set(s.dropna()))
reference_samples = samples_by_model.iloc[0]
for model, samples in samples_by_model.items():
    assert samples == reference_samples, f"Sample index mismatch for {model}"

print("Validation passed.")

## Model-Level Comparison

In [ ]:
model_summary_df = (
    length_df.groupby("model", as_index=False)
    .agg(
        n=("sample_index", "count"),
        avg_generated_tokens=("generated_tokens", "mean"),
        avg_gold_tokens=("gold_tokens", "mean"),
        avg_token_diff=("token_diff", "mean"),
        median_token_diff=("token_diff", "median"),
        avg_token_ratio=("token_ratio", "mean"),
        std_generated_tokens=("generated_tokens", "std"),
        min_generated_tokens=("generated_tokens", "min"),
        max_generated_tokens=("generated_tokens", "max"),
    )
)

round_cols = [
    "avg_generated_tokens",
    "avg_gold_tokens",
    "avg_token_diff",
    "median_token_diff",
    "avg_token_ratio",
    "std_generated_tokens",
]
model_summary_df[round_cols] = model_summary_df[round_cols].round(2)

model_summary_df["model"] = pd.Categorical(
    model_summary_df["model"],
    categories=[m for m in MODEL_ORDER if m != "Gold / Reference"],
    ordered=True,
)
model_summary_df = model_summary_df.sort_values("model").reset_index(drop=True)

display(model_summary_df)

In [ ]:
# Keep one gold-reference row per sample index.
gold_by_sample_df = (
    length_df[["sample_index", "reference_summary_zh", "gold_tokens"]]
    .drop_duplicates("sample_index")
    .sort_values("sample_index")
    .reset_index(drop=True)
)

gold_summary_df = pd.DataFrame(
    [
        {
            "Model": "Gold / Reference",
            "Generated summary length": round(gold_by_sample_df["gold_tokens"].mean(), 2),
            "n": int(gold_by_sample_df["gold_tokens"].count()),
            "std": round(gold_by_sample_df["gold_tokens"].std(ddof=1), 2),
            "median": round(gold_by_sample_df["gold_tokens"].median(), 2),
            "min": int(gold_by_sample_df["gold_tokens"].min()),
            "max": int(gold_by_sample_df["gold_tokens"].max()),
        }
    ]
)

generated_report_df = model_summary_df.rename(
    columns={
        "model": "Model",
        "avg_generated_tokens": "Generated summary length",
        "std_generated_tokens": "std",
        "min_generated_tokens": "min",
        "max_generated_tokens": "max",
    }
)[["Model", "Generated summary length", "n", "std", "min", "max"]]

report_table = pd.concat([gold_summary_df, generated_report_df], ignore_index=True)
report_table["Model"] = pd.Categorical(report_table["Model"], categories=MODEL_ORDER, ordered=True)
report_table = report_table.sort_values("Model").reset_index(drop=True)

display(report_table)

## Direct 300-Row Comparison

In [ ]:
direct_300_comparison_df = (
    length_df[length_df["source_group"] == "direct"]
    .sort_values(["sample_index", "model"])
    .reset_index(drop=True)
)

display(
    direct_300_comparison_df[
        [
            "sample_index",
            "model",
            "gold_tokens",
            "generated_tokens",
            "token_diff",
            "token_ratio",
            "reference_summary_zh",
            "generated_summary_zh",
        ]
    ]
)

## All-Model 400-Row Comparison

In [ ]:
all_model_comparison_df = (
    length_df.sort_values(["sample_index", "model"])
    .reset_index(drop=True)
)

display(
    all_model_comparison_df[
        [
            "sample_index",
            "model",
            "gold_tokens",
            "generated_tokens",
            "token_diff",
            "token_ratio",
        ]
    ]
)

## Sample-Level Wide Table

In [ ]:
wide_by_sample_df = length_df.pivot_table(
    index="sample_index",
    columns="model",
    values="generated_tokens",
    aggfunc="first",
).reset_index()

wide_by_sample_df = wide_by_sample_df.merge(
    gold_by_sample_df[["sample_index", "gold_tokens"]],
    on="sample_index",
    how="left",
)

wide_cols = ["sample_index", "gold_tokens", "mBART", "QWEN", "Gemma", "Aya-Expanse"]
wide_by_sample_df = wide_by_sample_df[[col for col in wide_cols if col in wide_by_sample_df.columns]]

display(wide_by_sample_df)

## Longest and Shortest Differences

In [ ]:
largest_over_gold_df = length_df.sort_values("token_diff", ascending=False).head(10)
largest_under_gold_df = length_df.sort_values("token_diff", ascending=True).head(10)

display(
    largest_over_gold_df[
        ["sample_index", "model", "gold_tokens", "generated_tokens", "token_diff", "token_ratio"]
    ]
)

display(
    largest_under_gold_df[
        ["sample_index", "model", "gold_tokens", "generated_tokens", "token_diff", "token_ratio"]
    ]
)

## Plot

In [ ]:
plot_df = report_table[["Model", "Generated summary length"]].copy()

ax = plot_df.set_index("Model")["Generated summary length"].plot(
    kind="barh",
    figsize=(9, 5),
    title="Average Summary Length Compared with Gold Reference",
)
ax.set_xlabel("Average summary tokens")
ax.invert_yaxis()

## Save Outputs

In [ ]:
model_summary_csv = OUTPUT_DIR / "model_gold_length_comparison.csv"
report_table_csv = OUTPUT_DIR / "report_generated_vs_gold_length.csv"
direct_300_csv = OUTPUT_DIR / "direct_300_gold_length_comparison.csv"
all_model_csv = OUTPUT_DIR / "all_models_gold_length_comparison_400rows.csv"
wide_csv = OUTPUT_DIR / "gold_length_by_sample_wide.csv"

model_summary_df.to_csv(model_summary_csv, index=False, encoding="utf-8-sig")
report_table.to_csv(report_table_csv, index=False, encoding="utf-8-sig")
direct_300_comparison_df.to_csv(direct_300_csv, index=False, encoding="utf-8-sig")
all_model_comparison_df.to_csv(all_model_csv, index=False, encoding="utf-8-sig")
wide_by_sample_df.to_csv(wide_csv, index=False, encoding="utf-8-sig")

print("Saved:", model_summary_csv)
print("Saved:", report_table_csv)
print("Saved:", direct_300_csv)
print("Saved:", all_model_csv)
print("Saved:", wide_csv)